## 面试问题

终止条件：完成/无进展/超预算/超时四类停止信号怎么排优先级？

## 回答主线

四类独立停止信号按优先级：完成 > 超时 > 超预算 > 无进展。完成即停(别浪费)，超时/超预算是硬约束(防失控)，无进展是软信号。本 Notebook 实现 `check_stop` 按固定优先级返回原因，构造四个分别触发一种信号的场景，再用「同时完成且超预算」的场景验证完成优先于止损。

## 真实案例

阈值：预算 100、超时 50、stall 3。构造 done/timeout/over_budget/stall 四个场景各触发一种信号，外加一个「已完成且已超预算」的场景。数据为教学状态，不代表真实系统。

In [1]:
LIMITS = {"budget": 100, "timeout": 50, "stall": 3}  # 定义预算、超时、stall 阈值。

scenarios = {  # 定义分别触发不同停止信号的场景。
    "done": {"done": True, "spent": 20, "elapsed": 10, "no_progress": 0},  # 已完成。
    "timeout": {"done": False, "spent": 20, "elapsed": 60, "no_progress": 0},  # 超时。
    "over_budget": {"done": False, "spent": 120, "elapsed": 10, "no_progress": 0},  # 超预算。
    "stall": {"done": False, "spent": 20, "elapsed": 10, "no_progress": 3},  # 无进展。
    "done_and_over_budget": {"done": True, "spent": 120, "elapsed": 10, "no_progress": 0},  # 同时完成且超预算。
}  # 结束场景定义。

print("阈值:", LIMITS)  # 展示阈值。
for name, s in scenarios.items():  # 逐个打印场景。
    print("  场景", name, ":", s)  # 展示每个场景状态。

阈值: {'budget': 100, 'timeout': 50, 'stall': 3}
  场景 done : {'done': True, 'spent': 20, 'elapsed': 10, 'no_progress': 0}
  场景 timeout : {'done': False, 'spent': 20, 'elapsed': 60, 'no_progress': 0}
  场景 over_budget : {'done': False, 'spent': 120, 'elapsed': 10, 'no_progress': 0}
  场景 stall : {'done': False, 'spent': 20, 'elapsed': 10, 'no_progress': 3}
  场景 done_and_over_budget : {'done': True, 'spent': 120, 'elapsed': 10, 'no_progress': 0}


## 基线（Baseline）

反面基线：只有一个粗糙信号（这里只在无进展极多时才停）。它既识别不了完成（做完不停、浪费），也不为超时/超预算止损（失控）。

In [2]:
def stop_only_steps(state, max_no_progress=999):  # 只看无进展步数模拟单一粗糙信号。
    if state["no_progress"] >= max_no_progress:  # 仅当无进展极多才停。
        return True, "max_steps"  # 单一信号停止。
    return False, "continue"  # 否则继续。

naive_stops = {name: stop_only_steps(s) for name, s in scenarios.items()}  # 用单一信号判断所有场景。
for name, r in naive_stops.items():  # 逐个打印。
    print("单一信号 场景", name, "->", r)  # 展示单一信号既不识别完成也不止损。

单一信号 场景 done -> (False, 'continue')
单一信号 场景 timeout -> (False, 'continue')
单一信号 场景 over_budget -> (False, 'continue')
单一信号 场景 stall -> (False, 'continue')
单一信号 场景 done_and_over_budget -> (False, 'continue')


## 失败案例与修正

单一信号该停不停。修正是 `check_stop` 按固定优先级依次检查完成/超时/超预算/无进展，命中即停并记录原因，其中完成优先于超预算。

In [3]:
def check_stop(state, limits):  # 按固定优先级检查四类停止信号。
    if state["done"]:  # 完成优先级最高。
        return True, "done"  # 完成即停即使超预算。
    if state["elapsed"] >= limits["timeout"]:  # 其次超时硬约束。
        return True, "timeout"  # 超时停止。
    if state["spent"] >= limits["budget"]:  # 再次超预算硬约束。
        return True, "over_budget"  # 超预算停止。
    if state["no_progress"] >= limits["stall"]:  # 最后无进展软信号。
        return True, "stall"  # 无进展干预。
    return False, "continue"  # 四类都不触发则继续。

In [4]:
decisions = {name: check_stop(s, LIMITS) for name, s in scenarios.items()}  # 用优先级信号判断所有场景。
for name, r in decisions.items():  # 逐个打印。
    print("优先级信号 场景", name, "->", r)  # 展示每个场景触发正确信号。

优先级信号 场景 done -> (True, 'done')
优先级信号 场景 timeout -> (True, 'timeout')
优先级信号 场景 over_budget -> (True, 'over_budget')
优先级信号 场景 stall -> (True, 'stall')
优先级信号 场景 done_and_over_budget -> (True, 'done')


In [5]:
priority_case = decisions["done_and_over_budget"]  # 取同时完成且超预算的场景结果。
print("同时完成且超预算 -> 判为:", priority_case[1], "(完成优先于止损)")  # 展示完成优先。
print("单一信号能识别完成吗:", naive_stops["done"][0])  # 展示单一信号识别不了完成。
print("优先级信号能识别完成吗:", decisions["done"][0])  # 展示优先级信号识别完成。

同时完成且超预算 -> 判为: done (完成优先于止损)
单一信号能识别完成吗: False
优先级信号能识别完成吗: True


## 结果解读

单一信号对所有场景都返回 continue，既不识别完成也不止损；优先级 `check_stop` 让四个场景各触发正确信号，且「完成且超预算」判为 done——成功优先于止损。要点：信号独立可测、优先级固定、停止必记原因。

In [6]:
assert decisions["done"] == (True, "done")  # 完成场景应判为 done。
assert decisions["timeout"] == (True, "timeout")  # 超时场景应判为 timeout。
assert decisions["over_budget"] == (True, "over_budget")  # 超预算场景应判为 over_budget。
assert decisions["stall"] == (True, "stall")  # 无进展场景应判为 stall。
assert decisions["done_and_over_budget"][1] == "done"  # 同时完成且超预算应完成优先。
assert naive_stops["done"][0] is False  # 单一信号无法识别完成。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
